# Statistics

`xftsim` ships a handful of per-generation statistic objects. Each one
implements the `xftsim.stats.Statistic` abstract base class. The
simulation runs each `Statistic` it's given at the end of every
generation; results are stored in `sim.results` as a list of
`GenerationResult` dataclasses.

What's changed from v0.3:

- The results container is now `sim.results: list[GenerationResult]`
  (one entry per completed generation), rather than the legacy
  `sim.results_store` dict-of-dicts. Each `GenerationResult` has a
  `generation` and a `statistics: dict[str, Any]`.
- The HE estimator no longer needs a sibling-pair filter — it now
  uses the GRM (genomic relationship matrix) and works on any sample,
  including the founder generation. The `randomized=True` flag of the
  legacy `HasemanElstonEstimator` has been replaced by a more general
  `n_probe=` parameter for stochastic trace estimation; the default
  (`n_probe=0`) builds the GRM exactly.
- GWAS now lives in its own module, `xftsim.gwas`.
- A new `ParentOffspringRegression` statistic estimates h² from
  parent-midparent regression slopes (requires a `TrioFilter`).
- `MatingStatistics` reads parent phenotypes from a `TrioView`
  (requires a `TrioFilter` to be active).

A small two-trait simulation we'll reuse:


In [ ]:
import xftsim as xft
import numpy as np

xft.config.print_durations_threshold = 10
np.random.seed(0)

N, M = 2000, 400
hap = xft.founders.founder_haplotypes_uniform_AFs(n=N, m=M)
eff_h = xft.effect.AdditiveEffects.from_h2(h2=0.5, m=M, seed=1)
eff_b = xft.effect.AdditiveEffects.from_h2(h2=0.4, m=M, seed=2)
arch = xft.arch.Architecture(
    formula='''
    height.G ~ genetic(eff_h)
    height.E ~ noise(0.5)
    height   ~ height.G + height.E
    BMD.G ~ genetic(eff_b)
    BMD.E ~ noise(0.6)
    BMD   ~ BMD.G + BMD.E
    ''',
    effects={'eff_h': eff_h, 'eff_b': eff_b},
)
rmap = xft.reproduce.RecombinationMap.from_haplotypes(hap, p=0.1)
mating = xft.mate.RandomMating(offspring_per_pair=2)

sim = xft.sim.Simulation(
    founder_haplotypes=hap, architecture=arch, recombination_map=rmap,
    mating_regime=mating, statistics=[], seed=42,
)


We didn't pass any statistics yet, so the simulation will just compute
phenotypes and mate each generation without recording anything beyond
the histories.

## Accessing results

After adding a `Statistic`, results accumulate in `sim.results`. We
can attach (or change) the statistic list at any time:


In [ ]:
sim.statistics = [xft.stats.SampleStatistics()]
sim.run(n_generations=2)

for r in sim.results:
    print(f'gen {r.generation}: {list(r.statistics.keys())}')


Each `GenerationResult.statistics` is keyed by the **class name** of
the `Statistic` that produced it:


In [ ]:
last = sim.results[-1]
ss = last.statistics['SampleStatistics']
list(zip(ss['keys'], np.diag(ss['cov'])))


## Skipping generations

We don't always want every statistic on every generation. Mutating
`sim.statistics` between `.run()` calls is the easiest way to control
this:


In [ ]:
sim2 = xft.sim.Simulation(
    founder_haplotypes=hap, architecture=arch, recombination_map=rmap,
    mating_regime=mating, statistics=[xft.stats.SampleStatistics()],
    seed=42,
)
sim2.run(n_generations=1)
sim2.statistics = []
sim2.run(n_generations=3)
sim2.statistics = [xft.stats.SampleStatistics()]
sim2.run(n_generations=1)
[(r.generation, list(r.statistics.keys())) for r in sim2.results]


## `SampleStatistics`

Returns a dict with three keys per generation:

- `cov` — `k × k` empirical phenotype covariance matrix
- `var` — diagonal of `cov` (per-component variances)
- `keys` — list of phenotype-component names matching the matrix axes


In [ ]:
ss = sim.results[-1].statistics['SampleStatistics']
ss['keys'], ss['cov'].shape


## `MatingStatistics`

Returns:

- `n_mating_pairs` — number of unique parent pairs that produced the
  current generation
- `mean_offspring_count` — mean offspring count per pair
- `spouse_correlations` — dict `phenotype_name -> Pearson r` across
  unique mating pairs

`spouse_correlations` is computed via the `TrioView` from a
`TrioFilter`. At generation 0 (founders) the TrioView is empty, so
the dict will be empty; gen-1 results reflect the founder mating.


In [ ]:
sim3 = xft.sim.Simulation(
    founder_haplotypes=hap, architecture=arch, recombination_map=rmap,
    mating_regime=xft.mate.LinearAssortativeMating(
        component_names=['height', 'BMD'], r=0.3,
    ),
    statistics=[xft.stats.MatingStatistics()],
    filters={'trio': xft.filters.TrioFilter()},
    seed=42,
)
sim3.run(n_generations=2)
sim3.results[-1].statistics['MatingStatistics']['spouse_correlations']


## Heritability and genetic correlation: `HasemanElstonEstimator`

The HE estimator now uses the GRM (genomic relationship matrix)
formula

$$\hat{\text{cov}}_g = \frac{Y^\top (K Y - Y)}{\text{tr}(K^2) - n}$$

with $K = G G^\top / m$ from per-SNP standardised genotypes. It works
at any generation — including founders — and on any sample (no
sibling pairs required).


In [ ]:
sim4 = xft.sim.Simulation(
    founder_haplotypes=hap, architecture=arch, recombination_map=rmap,
    mating_regime=mating,
    statistics=[xft.stats.HasemanElstonEstimator(
        phenotype_keys=['height', 'BMD'],
    )],
    seed=42,
)
sim4.run(n_generations=1)
he = sim4.results[-1].statistics['HasemanElstonEstimator']
print('h2 estimates:')
for k in ['height', 'BMD']:
    print(f'  {k}: {he[k]["h2"]:.3f}')
print('genetic covariance matrix:')
print(he['_cov_g'])


For large `n` the deterministic exact trace `tr(K²)` builds the
`(n, m)` standardized matrix in memory. Set `n_probe > 0` to use
Hutchinson's stochastic trace estimator instead:


In [ ]:
he_stoch = xft.stats.HasemanElstonEstimator(
    phenotype_keys=['height', 'BMD'],
    n_probe=64,
)


## Parent-offspring regression: `ParentOffspringRegression`

Regresses offspring phenotype on midparent value; under an additive
model the slope estimates h². Requires a `TrioFilter`:


In [ ]:
sim5 = xft.sim.Simulation(
    founder_haplotypes=hap, architecture=arch, recombination_map=rmap,
    mating_regime=mating,
    statistics=[xft.stats.ParentOffspringRegression(filter_name='trio')],
    filters={'trio': xft.filters.TrioFilter()},
    seed=42,
)
sim5.run(n_generations=2)
last = sim5.results[-1].statistics.get('ParentOffspringRegression', {})
for k, v in last.items():
    print(f'{k}: h2 = {v["h2"]:.3f}, slope = {v["slope"]:.3f}, se = {v["se"]:.3f}, n_trios = {v["n_trios"]}')


## GWAS

GWAS has moved out of `xftsim.stats` and into its own module,
`xftsim.gwas`. See the API reference for the current set of routines
(sumstats, polygenic-index estimation, cross-validated PGI scoring).

## Filters

The `xftsim.filters` module provides views over the current
generation that statistics can consume:

| Filter | View | Use |
| --- | --- | --- |
| `TrioFilter` | `TrioView` | Parent-offspring trios (needed by `MatingStatistics`, `ParentOffspringRegression`) |
| `SibPairFilter` | `SibPairView` | Vectorised sibling pairs |
| `UnrelatedFilter` | `UnrelatedView` | Greedy IBD-pruned unrelated set |
| `AscertainmentFilter` | `AscertainedView` | Trait-based ascertainment |
| `SubsampleFilter` | `SubsampleView` | Random subsample |

You pass them as a `dict[str, Filter]` to `Simulation(filters=...)`.
The named filter views are then available to each statistic.

## Writing your own statistic

Subclass `xftsim.stats.Statistic` and implement `.estimate()`:


In [ ]:
class MyMeanPhenotypeStat(xft.stats.Statistic):
    def estimate(self, phenotype_history, filtered_views, generation, **kwargs):
        if generation not in phenotype_history:
            return None
        pheno = phenotype_history[generation]
        return {k: float(np.mean(pheno[k])) for k in pheno.keys}

sim6 = xft.sim.Simulation(
    founder_haplotypes=hap, architecture=arch, recombination_map=rmap,
    mating_regime=mating,
    statistics=[MyMeanPhenotypeStat()],
    seed=42,
)
sim6.run(n_generations=1)
sim6.results[-1].statistics


Note the key in `result.statistics` is the class name
(`'MyMeanPhenotypeStat'`).
